# **NLP입문**

## **0. 환경 설정**

- 그래프에 한글이 깨지지 않도록 나눔 폰트를 설치
- 실습 전체에서 사용할 라이브러리를 불러오기



In [1]:
# 한글 설치
# !pip install -q koreanize-matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import os, re, warnings
import numpy as np
import pandas as pd
from collections import Counter

## **1. Corpus 준비하기**

이번 실습에서는 **한국어 영화 리뷰 120개**로 이루어진 아주 작은 코퍼스를 사용한다.

| 컬럼 | 설명 |
|---|---|
| `review` | 리뷰 문장 (원문 텍스트) |
| `label` | 감성 라벨 — 1 = 긍정, 0 = 부정 |

- 입력(특징, feature) 으로 사용하는 것 : `review` 컬럼의 **문장 전체**
- 출력(종속변수, target) 으로 사용하는 것 : `label` 컬럼
- 문장은 **비정형 데이터**이므로 그대로는 계산할 수 없다.
- 문장을 숫자로 바꾸는 전처리를 해보자.


In [3]:
POS = [
"이 영화 정말 최고였어요 배우들 연기가 훌륭합니다",
"스토리가 탄탄하고 연출이 훌륭해서 시간 가는 줄 몰랐다",
"올해 본 영화 중에 단연 최고 강력 추천합니다",
"음악이 아름답고 영상미가 정말 훌륭했어요",
"배우들 연기가 자연스러워서 몰입이 잘 되었다",
"가족과 같이 보기 좋은 따뜻한 영화 추천해요",
"두 번 세 번 봐도 재미있는 명작입니다",
"결말이 감동적이어서 눈물이 났어요 정말 좋았습니다",
"연출과 각본 모두 훌륭한 수작 강력 추천",
"기대 이상으로 재미있었고 배우들 연기도 좋았다",
"유머와 감동이 적절히 섞여 있어서 즐겁게 봤어요",
"소재가 참신하고 전개가 빨라서 지루하지 않았다",
"촬영이 아름답고 배경음악이 좋아서 여운이 남는다",
"아이들과 함께 보기에 정말 좋은 작품이었어요",
"배우 연기력이 대단해서 영화에 완전히 몰입했다",
"생각할 거리를 주는 좋은 영화 추천하고 싶어요",
"반전이 훌륭해서 끝까지 긴장하며 봤습니다",
"감독의 연출이 섬세하고 이야기가 따뜻해서 좋았다",
"웃다가 울다가 정말 즐거운 시간이었어요",
"완성도가 높고 재미까지 있어서 만족스러운 영화",
"돈이 아깝지 않은 영화 다시 보고 싶다",
"배우 연기와 음악이 모두 훌륭한 명작이었다",
"이야기가 신선하고 결말도 만족스러웠습니다",
"오랜만에 만난 훌륭한 작품 정말 추천합니다",
"시간 가는 줄 모르고 재미있게 관람했어요",
"연기와 연출이 모두 좋아서 몰입감이 뛰어났다",
"영상미가 뛰어나고 음악도 아름다워서 감동했어요",
"각본이 탄탄해서 끝까지 몰입해서 봤습니다",
"주인공의 성장이 감동적이어서 여운이 오래 남았다",
"편집이 깔끔하고 전개가 빨라서 지루할 틈이 없었다",
"기대하지 않았는데 예상 밖으로 훌륭한 영화였어요",
"연출이 세련되고 대사가 좋아서 인상 깊었다",
"배우들의 호흡이 좋아서 보는 내내 즐거웠습니다",
"따뜻한 메시지가 있어서 마음이 편안해지는 작품",
"긴장감이 끝까지 유지되어서 정말 재미있었어요",
"음향과 미술이 훌륭해서 극장에서 볼 가치가 있다",
"캐릭터가 매력적이어서 계속 응원하게 되는 영화",
"결말이 깔끔해서 만족스럽게 극장을 나왔습니다",
"소재는 무겁지만 연출이 따뜻해서 좋았어요",
"대사 하나하나가 좋아서 몇 번이나 곱씹게 된다",
"재미와 감동을 모두 잡은 훌륭한 작품이었습니다",
"연기 연출 음악 모두 뛰어난 완성도 높은 영화",
"보고 나서 기분이 좋아지는 유쾌한 작품이에요",
"장면 하나하나가 아름다워서 눈이 즐거웠습니다",
"이야기 구성이 치밀해서 끝까지 몰입할 수 있었다",
"가볍게 보기 좋은 유쾌하고 재미있는 영화였어요",
"배우의 연기 변신이 놀라워서 감탄하며 봤다",
"올해 최고의 수작 주변에 강력히 추천하고 싶다",
"잔잔하지만 여운이 깊게 남는 좋은 작품입니다",
"음악이 영화와 잘 어울려서 감동이 배가 되었어요",
"연출과 연기가 조화로워서 완성도가 매우 높았다",
"오랜만에 극장에서 크게 웃고 나온 즐거운 영화",
"세심한 연출 덕분에 인물들이 살아 있는 느낌이다",
"결말이 예상과 달라서 신선하고 재미있었습니다",
"영화가 끝나고도 계속 생각나는 훌륭한 작품",
"연기가 진솔해서 인물에게 깊이 공감할 수 있었다",
"짜임새 있는 각본과 좋은 연기가 돋보이는 영화",
"화면이 아름답고 이야기도 따뜻해서 추천합니다",
"시간이 아깝지 않은 재미있고 감동적인 영화였다",
"기대 이상의 완성도 다시 보고 싶은 작품이에요",
]

NEG = [
"스토리가 지루하고 전개가 너무 느려서 실망했다",
"연기가 어색해서 몰입이 전혀 되지 않았습니다",
"돈이 아까운 영화 절대 추천하지 않아요",
"결말이 허무해서 보고 나서 화가 났습니다",
"각본이 엉망이고 연출도 별로였다",
"기대했는데 너무 실망스러운 작품이었어요",
"배우 연기가 부자연스럽고 대사가 유치했다",
"두 시간이 너무 길게 느껴진 지루한 영화",
"소재는 좋았지만 전개가 산만해서 별로였어요",
"음악도 영상도 촌스럽고 재미가 없었다",
"스토리에 개연성이 없어서 보다가 잠들었습니다",
"최악의 영화 시간이 정말 아까웠어요",
"유치한 대사와 뻔한 전개 실망이 컸다",
"배우들 연기는 좋았지만 각본이 너무 별로였다",
"보는 내내 지루해서 중간에 나오고 싶었어요",
"개연성 없는 전개에 결말까지 허무했습니다",
"졸작이라는 말이 어울리는 실망스러운 영화",
"홍보만 화려하고 내용은 텅 빈 영화였다",
"연출이 산만하고 편집도 엉망이라 별로였어요",
"재미도 감동도 없어서 추천하지 않습니다",
"캐릭터가 매력이 없어서 몰입이 안 되었다",
"전개가 뻔해서 결말이 다 예상되어 지루했어요",
"기대가 컸던 만큼 실망도 큰 작품이었습니다",
"대사가 어색하고 연기도 부족해서 아쉬웠다",
"다시는 보고 싶지 않은 최악의 영화였어요",
"편집이 엉성해서 이야기를 따라가기 힘들었다",
"음향이 어수선하고 화면도 답답해서 별로였다",
"인물들의 행동에 설득력이 없어서 몰입이 깨졌어요",
"러닝타임이 길기만 하고 알맹이가 없는 영화",
"억지 감동을 강요하는 전개라 불편했습니다",
"연출이 낡아서 요즘 영화 같지 않고 지루했다",
"주인공의 선택이 이해되지 않아 답답했어요",
"대사가 촌스럽고 유머도 전혀 웃기지 않았다",
"돈과 시간을 모두 낭비한 기분이 드는 작품",
"스토리가 산만해서 무슨 말을 하려는지 모르겠다",
"기대작이라더니 실망만 안겨준 영화였습니다",
"결말이 급하게 마무리되어 허무하고 아쉬웠다",
"캐릭터 설정이 어색해서 공감하기 어려웠어요",
"긴장감이 전혀 없어서 보다가 졸았습니다",
"영상은 그럴듯한데 이야기가 비어 있어서 별로",
"뻔한 클리셰만 반복되어 지루하기 짝이 없었다",
"연기 톤이 들쭉날쭉해서 몰입이 계속 끊겼어요",
"내용이 불친절해서 이해하기 어렵고 답답했다",
"감동을 노린 장면이 오히려 어색하게 느껴졌다",
"장면 전환이 거칠어서 보기 불편한 영화였어요",
"주제 의식이 얕아서 남는 것이 없는 작품",
"광고와 다르게 재미없고 실망스러운 영화였다",
"배경음악이 과해서 대사가 잘 들리지 않았어요",
"이야기가 늘어져서 후반부는 참고 봤습니다",
"인물 관계가 정리되지 않아 끝까지 혼란스러웠다",
"연출이 과장되어 오히려 몰입을 방해했어요",
"결말이 어이없어서 극장을 나오며 한숨이 나왔다",
"소재만 좋고 완성도는 낮은 아쉬운 작품입니다",
"반전이 억지스러워서 전혀 놀랍지 않았어요",
"대사와 상황이 촌스러워서 보기 민망했다",
"기대했던 배우의 연기마저 실망스러웠습니다",
"전개가 느리고 지루해서 시간이 아까웠어요",
"작품성도 재미도 없어서 추천할 수 없는 영화",
"편집이 어색하고 흐름이 끊겨서 몰입이 안 됐다",
"두 번은 절대 보고 싶지 않은 실망스러운 영화",
]

In [4]:
reviews = [(t,1) for t in POS] + [(t,0) for t in NEG]
print(*reviews[0:8], sep="\n")
print(*reviews[-1:-8:-1], sep='\n')

('이 영화 정말 최고였어요 배우들 연기가 훌륭합니다', 1)
('스토리가 탄탄하고 연출이 훌륭해서 시간 가는 줄 몰랐다', 1)
('올해 본 영화 중에 단연 최고 강력 추천합니다', 1)
('음악이 아름답고 영상미가 정말 훌륭했어요', 1)
('배우들 연기가 자연스러워서 몰입이 잘 되었다', 1)
('가족과 같이 보기 좋은 따뜻한 영화 추천해요', 1)
('두 번 세 번 봐도 재미있는 명작입니다', 1)
('결말이 감동적이어서 눈물이 났어요 정말 좋았습니다', 1)
('두 번은 절대 보고 싶지 않은 실망스러운 영화', 0)
('편집이 어색하고 흐름이 끊겨서 몰입이 안 됐다', 0)
('작품성도 재미도 없어서 추천할 수 없는 영화', 0)
('전개가 느리고 지루해서 시간이 아까웠어요', 0)
('기대했던 배우의 연기마저 실망스러웠습니다', 0)
('대사와 상황이 촌스러워서 보기 민망했다', 0)
('반전이 억지스러워서 전혀 놀랍지 않았어요', 0)


## **2. Tokenization**

문장을 작은 단위로 나누는 과정이다.

* 띄어쓰기 기준으로 자연어를 나눠보자.
* 당연히 라벨을 버리면 안된다.
* 새로운 list를 하나 선언하고 ([잘린 단어들], label) 형식으로 넣어두자

In [5]:
# tokens = []
# for r in reviews:
#     tokens.append((r[0].split(), r[1])) # 앞의 문장
# print(tokens)

tokens = []
for text, label in reviews:
    tokens.append((text.split(), label))
    
print(*tokens[:3], sep='\n')

(['이', '영화', '정말', '최고였어요', '배우들', '연기가', '훌륭합니다'], 1)
(['스토리가', '탄탄하고', '연출이', '훌륭해서', '시간', '가는', '줄', '몰랐다'], 1)
(['올해', '본', '영화', '중에', '단연', '최고', '강력', '추천합니다'], 1)


## **3. 학습, 테스트 데이터 분리하기**
- 모델은 Train 데이터로 학습하고 Test 데이터로 성능을 확인한다.
- Vocabulary도 Train 데이터만 사용하여 만든다.
- 20%를 테스트 데이터로 활용한다.
- sklearn의 train_test_split을 활용한다.
- Tokenization 과정에서 데이터와 라벨이 같이 들어있다는것을 고려하자.

In [6]:
from sklearn.model_selection import train_test_split

labels = [label for _, label in tokens]

train_data, test_data = train_test_split(
    tokens,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print("학습 데이터:", len(train_data))
print("테스트 데이터:", len(test_data))
print("학습 데이터 예시:", train_data[0])

학습 데이터: 96
테스트 데이터: 24
학습 데이터 예시: (['소재가', '참신하고', '전개가', '빨라서', '지루하지', '않았다'], 1)


## **4. Vocabulary**
각 토큰에 번호를 부여한다.
* 토큰화한 단어는 아직 문자열이기 때문에 그대로 계산할 수 없다.
* 각 단어에 고유한 번호(Token ID)를 부여한다.

In [7]:
vocab = {}

for statement, label in train_data:
    for word in statement:
        if word not in vocab:
            vocab[word] = len(vocab) + 1

print(vocab)

{'소재가': 1, '참신하고': 2, '전개가': 3, '빨라서': 4, '지루하지': 5, '않았다': 6, '홍보만': 7, '화려하고': 8, '내용은': 9, '텅': 10, '빈': 11, '영화였다': 12, '세심한': 13, '연출': 14, '덕분에': 15, '인물들이': 16, '살아': 17, '있는': 18, '느낌이다': 19, '배우': 20, '연기가': 21, '부자연스럽고': 22, '대사가': 23, '유치했다': 24, '배우들의': 25, '호흡이': 26, '좋아서': 27, '보는': 28, '내내': 29, '즐거웠습니다': 30, '이야기': 31, '구성이': 32, '치밀해서': 33, '끝까지': 34, '몰입할': 35, '수': 36, '있었다': 37, '이': 38, '영화': 39, '정말': 40, '최고였어요': 41, '배우들': 42, '훌륭합니다': 43, '기대': 44, '이상의': 45, '완성도': 46, '다시': 47, '보고': 48, '싶은': 49, '작품이에요': 50, '느리고': 51, '지루해서': 52, '시간이': 53, '아까웠어요': 54, '자연스러워서': 55, '몰입이': 56, '잘': 57, '되었다': 58, '연기': 59, '톤이': 60, '들쭉날쭉해서': 61, '계속': 62, '끊겼어요': 63, '소재는': 64, '무겁지만': 65, '연출이': 66, '따뜻해서': 67, '좋았어요': 68, '이야기가': 69, '신선하고': 70, '결말도': 71, '만족스러웠습니다': 72, '음향이': 73, '어수선하고': 74, '화면도': 75, '답답해서': 76, '별로였다': 77, '유머와': 78, '감동이': 79, '적절히': 80, '섞여': 81, '있어서': 82, '즐겁게': 83, '봤어요': 84, '연기력이': 85, '대단해서': 86, '영화에': 87, '완전히': 88, '몰입했다': 89, '각본이': 

## **5. 특수 토큰**
- 문장을 숫자로 바꾸려면 두 가지 상황을 추가로 처리해야 한다.
- `<PAD>` : 서로 다른 문장의 길이를 맞출 때 사용한다.
- `<UNK>` : Vocabulary에 없는 단어가 들어왔을 때 사용한다.

`0`과 `1`을 두 특수 토큰에 사용하기 위해 기존 단어의 번호를 2씩 증가시킨다.

In [8]:
# 기존 ID를 1씩 증가
vocab = {word: idx + 1 for word, idx in vocab.items()}

vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

## **6. Encoding**
* Vocabulary를 이용하여 각 단어를 Token ID로 변환한다.
* Vocabulary에 없는 단어는 `<UNK>`의 번호인 `1`로 변환한다.

In [9]:
def encoding(data):
    # 현재 데이터에서 모든 문장들을 우리가 부여한 ID
    # vocab에 해당하는 숫자들로 변환해서 return을 해주는 함수

    result = []
    for words, label in data:
        encoded = [vocab.get(word, vocab['<UNK>']) for word in words]
                # for word in words:
                #     if word in vocab:
                #         encoded.append(vocab[word])
                #     else:
                #         encoded.append(vocab['<UNK>'])
                
        result.append((encoded, label))
    return result

train_encoded = encoding(train_data)
test_encoded = encoding(test_data)

print("학습 데이터")
print("변환 전:", train_data[0])
print("변환 후:", train_encoded[0])

print("\n테스트 데이터")
print("변환 전:", test_data[0])
print("변환 후:", test_encoded[0])

학습 데이터
변환 전: (['소재가', '참신하고', '전개가', '빨라서', '지루하지', '않았다'], 1)
변환 후: ([2, 3, 4, 5, 6, 7], 1)

테스트 데이터
변환 전: (['스토리가', '탄탄하고', '연출이', '훌륭해서', '시간', '가는', '줄', '몰랐다'], 1)
변환 후: ([326, 1, 67, 1, 370, 371, 372, 1], 1)


## **7. Padding**
* 문장마다 Token의 개수가 다르기 때문에 하나의 Batch로 묶으려면 길이를 맞춰야 한다.
* 부족한 부분은 `<PAD>`의 번호인 `0`으로 채운다.
* 각 문장의 실제 마지막 단어 위치를 찾기 위해 Padding 전 문장 길이도 함께 저장한다.

In [10]:
max_len = max(len(word) for word, _ in train_encoded)
print("최대 문장 길이 :", max_len)

def add_padding(data):
    result = []

    for word, label in data:
        word = word[:max_len]  # 최대 길이 초과 시 자르기

        cur_len = len(word)
        num_paddings = max_len - cur_len

        word = word + [vocab['<PAD>']] * num_paddings

        result.append((word, cur_len, label)) # padding 작업 완료된 token, 

    return result


train_padded = add_padding(train_encoded)
test_padded = add_padding(test_encoded)
print(train_padded[0])
print(test_padded[0])

최대 문장 길이 : 8
([2, 3, 4, 5, 6, 7, 0, 0], 6, 1)
([326, 1, 67, 1, 370, 371, 372, 1], 8, 1)


## **8. Tensor 변환**
* Padding이 끝난 데이터를 PyTorch Tensor로 변환한다.
* 각 문장의 실제 마지막 단어 위치를 찾기 위해 Padding 전 문장 길이도 함께 저장한다.

In [11]:
import torch

X_train = torch.tensor([word for word, _, _ in train_padded])           # 패딩된 토큰 ID
train_length = torch.tensor([length for _, length, _ in train_padded])  # 실제 문장 길이
y_train = torch.tensor([label for _, _, label in train_padded])         # 정답 : 긍정(1) / 부정(0)

X_test = torch.tensor([word for word, _, _ in test_padded])
test_length = torch.tensor([length for _, length, _ in test_padded])
y_test = torch.tensor([label for _, _, label in test_padded])

**데이터, 길이, 라벨을 합쳐 dataset으로 변환한다.**
```
from torch.utils.data import TensorDataset
```

In [12]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train, train_length, y_train)
test_dataset = TensorDataset(X_test, test_length, y_test)

next(iter(train_dataset))

(tensor([2, 3, 4, 5, 6, 7, 0, 0]), tensor(6), tensor(1))

**Batch Size를 32로 설정하여 loader를 생성한다.**
```
from torch.utils.data import DataLoader
```

In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

## **9. Embedding + 모델링**
Token ID는 단어를 구분하기 위한 번호일 뿐 숫자의 크기에는 의미가 없다.

- `nn.Embedding`을 이용하여 각 Token ID를 여러 개의 값을 가진 Vector로 변환한 후 LSTM에 입력한다.
- `Token ID → Embedding → LSTM → Linear → 긍정/부정`

In [14]:
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        # 1. embedding layer - ID로 인코딩 된 문자열 - 벡터화
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size,
            padding_idx = vocab['<PAD>']
        )

        # 2. SLTM 레이어
        self.lstm = nn.LSTM(
            input_size = embedding_size,
            hidden_size = hidden_size
        )


        # 3. 출력 레이어
        self.linear = nn.Linear(hidden_size, 2)


    def forward(self, X, length):
        z = self.embedding(X)
        hidden, _ = self.lstm(z)
            # hidden : 은닉상태 /  _ : 은닉 cell의 정보
            # hidden에는 각 단어들을 처리했을 떄의 모든 (기억) / 은닉 상태가 모두 포함되어있다

        # many-to-one model
        batch_index = torch.arange(z.size(0))           # [1,2,3,0,0,0] -> [0,1,2]
        last_hidden = hidden[batch_index, length -1]    # 각 시점의 마지막 기억들만 학습

        # 마지막 실제 토큰의 은닉 상태만 선택
        return self.linear(last_hidden)


model = LSTM(
    vocab_size = len(vocab),
    embedding_size = 16,
    hidden_size = 32
)

model.eval()
embedded = model.embedding(X_train[:1])
print(embedded[0])


tensor([[-0.1226, -0.6766,  0.3793, -0.0807, -0.5464, -1.8133, -0.1481,  0.2374,
         -0.3334, -0.5687,  0.2360, -0.8432,  0.2154, -1.0532,  1.0039,  0.0100],
        [ 0.2269, -0.6827,  1.6686,  0.8605, -0.3665,  0.1868, -1.7173,  2.1391,
         -0.8189, -1.9041,  0.4821,  0.3478, -0.3197, -1.0236,  1.4663, -1.5982],
        [-0.2513,  0.0562, -0.3276,  0.5518, -0.0862, -0.6103, -0.2806, -0.3526,
         -2.0846,  0.5906, -0.8573,  0.2269, -1.6288,  0.1403,  0.4240, -0.5111],
        [ 1.2526,  2.3541,  0.7456,  0.5324, -0.2191, -0.2816, -0.2493, -0.0793,
          0.3534,  0.3977,  0.5396, -0.7079, -1.0972, -0.0316, -0.9832,  0.4098],
        [ 1.5043, -0.7355, -0.8154, -1.4577,  2.1990, -1.4935, -1.3236, -0.0969,
         -0.2474, -2.0171,  0.5306,  0.1163, -0.0423, -1.5609,  0.5533,  0.8467],
        [-0.4567, -0.1853, -1.2807, -0.7468, -0.7057,  0.7593, -1.2853,  1.9124,
          1.0010,  0.8129,  2.0999,  1.6137,  1.9368, -1.2339, -2.2851, -0.2294],
        [ 0.0000,  0.0

Embedding을 통과하면 각 Token이 Vector로 변환된다.

예를 들어 `embedding_size=16`이면 입력 형태는 `[Batch, Timestep]`에서 `[Batch, Timestep, 16]`으로 바뀐다.

In [15]:
# Batch 데이터 하나 가져오기
X_batch, length_batch, y_batch = next(iter(train_loader))

# Embedding 적용
embedded = model.embedding(X_batch)

print("입력 Shape:", X_batch.shape)
print("Embedding Shape:", embedded.shape)
print("Embedding 결과:", embedded[0])

입력 Shape: torch.Size([32, 8])
Embedding Shape: torch.Size([32, 8, 16])
Embedding 결과: tensor([[-0.0396, -0.4873,  0.9901, -0.7036,  0.5968, -0.9266,  0.4321, -0.2953,
         -1.4202,  1.4589,  1.2838, -0.0027, -1.1725,  0.6715,  0.2774, -1.6737],
        [ 2.4193, -0.7849,  1.6425, -0.5169,  0.4186, -2.3117, -0.4664, -0.1587,
          0.2190,  0.1069,  0.3226,  1.4257,  0.0296, -1.0429,  1.7890,  0.2717],
        [-2.2582, -0.1066, -0.1956, -0.2936,  0.1891,  0.4637,  0.7007,  0.4875,
          0.0867,  1.2527, -0.3871,  0.5089, -0.8440, -1.7572,  0.9415,  0.8259],
        [ 0.0435,  0.2758,  0.1731,  0.0286, -0.2303, -0.0913,  0.2188, -0.3436,
         -0.4443, -1.3976, -0.5407, -0.0977,  1.3138,  0.8903, -0.4183,  0.2090],
        [ 0.7645, -0.1883, -0.8238, -0.3673,  0.0786,  1.1263, -0.1936,  0.8063,
          1.3607,  0.1871, -0.0877, -1.4666, -0.3409,  1.7240, -2.0056,  0.4510],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  

## **10. 모델 학습**
- 출력은 부정(0), 긍정(1) 두 개의 클래스이므로 `CrossEntropyLoss`를 사용한다.

In [25]:
import torch.optim as optim

# 손실 함수
criterion = nn.CrossEntropyLoss()

# 최적화 알고리즘
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for X_batch, length_batch, y_batch in train_loader:

        optimizer.zero_grad()
        outputs = model(X_batch, length_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()    # 가중치 업데이트

        total_loss += loss.item()

    if (epoch + 1) % 15 == 0:
        print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")

Epoch 15: Loss = 0.1909
Epoch 30: Loss = 0.1716


## **11. 모델 평가**
* 학습에 사용하지 않은 Test 데이터의 감성을 예측하고 정확도를 확인한다.
* 데이터가 매우 작고 띄어쓰기만으로 토큰화했기 때문에 정확도는 높지 않을 수 있다.

In [26]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X_batch, length_batch, y_batch in test_loader:

        outputs = model(X_batch, length_batch)  # 모델 예측

        predicted = outputs.argmax(dim=1)       # 두 클래스 중 점수가 높은 것 선택
        
        correct += (predicted == y_batch).sum().item()  # 정답 개수 계산
        total += y_batch.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.2%}")

Test Accuracy: 37.50%


## **12. 새로운 문장 예측**
- 새로운 문장도 학습 데이터와 같은 순서로 처리한다.
- `Tokenization → Integer Encoding → Padding → Tensor → Model`

In [23]:
def predict(statement):
    # 1. Tokenization
    tokens = statement.split()

    if not tokens:
        raise ValueError("문장을 입력하세요.")

    # 2. Integer Encoding
    word = [vocab.get(t, vocab['<UNK>']) for t in tokens]

    # 3. Padding
    word = word[:max_len]
    word_length = len(word)
    word = word + [vocab['<PAD>']] * (max_len - word_length)

    # 4. Tensor 변환
    x = torch.tensor([word])
    length = torch.tensor([word_length])

    # 5. 모델 예측
    model.eval()
    with torch.no_grad():
        outputs = model(x, length)
        pred = outputs.argmax(dim=1).item()

    return "긍정" if pred == 1 else "부정"

print(predict("이 영화 정말 재미있어요"))
print(predict("너무 지루하고 재미없는 영화"))
print(predict("배우들의 연기가 훌륭했어요"))

부정
긍정
부정
